# 🚀 HUẤN LUYỆN CHUYÊN SÂU YOLOPv2 TRÊN TOÀN BỘ 3.626 MẪU TUSIMPLE & VIỆT NAM
### ĐỀ TÀI: HỆ THỐNG PHÁT HIỆN LÀN ĐƯỜNG & PHƯƠNG TIỆN GIAO THÔNG
**Nhóm 1 — Lớp 24DHTT02 | HK7 (2026–2027)**

---
### 📋 HƯỚNG DẪN 3 BƯỚC ĐƠN GIẢN:
1. **Bước 1**: Nhìn sang cột bên trái của Google Colab, bấm vào biểu tượng 📁 **Files**.
2. **Bước 2**: Kéo thả tệp **`tusimple_colab_bundle.zip`** (khoảng ~485 MB từ máy tính của bạn) vào đó.
3. **Bước 3**: Chọn menu **Runtime (Thời gian chạy) ➔ Run all (Chạy tất cả)**.

> ⚡ **Hệ thống sẽ tự động hoàn toàn**:
> - Giải nén toàn bộ 3.626 mẫu TuSimple đã phân chia sẵn Train (2.900), Val (362), Test (364).
> - Tự động tải file trọng số gốc `yolopv2.pt` (156 MB) từ GitHub Release siêu tốc.
> - Áp dụng chiến lược **Freeze 100% Backbone**, chỉ huấn luyện 2 nhánh Head vạch đường và làn xe với Focal + Dice Loss.
> - Vẽ biểu đồ Loss/Metrics chuẩn báo cáo và **TỰ ĐỘNG TẢI FILE TRỌNG SỐ VỀ MÁY CỦA BẠN** khi kết thúc!

## 1. Kiểm tra cấu hình phần cứng GPU

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## 2. Giải nén Bộ Dữ Liệu TuSimple & Tải Trọng số Gốc

In [ ]:
import os
from pathlib import Path

# 1. Giải nén gói mã nguồn & dữ liệu TuSimple
if Path('/content/tusimple_colab_bundle.zip').exists():
    print('📦 Tìm thấy tusimple_colab_bundle.zip! Đang giải nén...')
    !unzip -q -o /content/tusimple_colab_bundle.zip -d /content/
    print('✓ Giải nén thành công toàn bộ dữ liệu 3.626 mẫu TuSimple và mã nguồn!')
elif Path('/content/colab_training_bundle.zip').exists():
    !unzip -q -o /content/colab_training_bundle.zip -d /content/
    print('✓ Giải nén colab_training_bundle.zip thành công!')
else:
    print('ℹ️ Vui lòng kéo thả file tusimple_colab_bundle.zip vào cột Files bên trái trước khi chạy tiếp.')

# 2. Tự động tải file trọng số yolopv2.pt gốc từ GitHub Release nếu chưa có
!mkdir -p weights
if not Path('weights/yolopv2.pt').exists() or Path('weights/yolopv2.pt').stat().st_size < 100000000:
    print('📥 Đang tự động tải trọng số yolopv2.pt gốc từ GitHub Release...')
    !wget -q --show-progress -O weights/yolopv2.pt https://github.com/CAIC-AD/YOLOPv2/releases/download/V0.0.1/yolopv2.pt
    print('✓ Tải trọng số gốc yolopv2.pt thành công!')
else:
    print('✓ Đã có sẵn file trọng số weights/yolopv2.pt.')

## 3. Cài đặt các thư viện phụ thuộc

In [ ]:
!pip install -q ultralytics torchvision opencv-python matplotlib scipy

## 4. Xác nhận Số lượng Dữ liệu TuSimple

In [ ]:
from pathlib import Path

data_dir = Path('data/dataset')
print("Kiểm tra các thư mục dữ liệu đã sẵn sàng:")
for sp in ['train', 'val', 'test']:
    n_img = len(list((data_dir / sp / 'images').glob('*.*')))
    n_da = len(list((data_dir / sp / 'drivable').glob('*.png')))
    n_ll = len(list((data_dir / sp / 'lane').glob('*.png')))
    print(f"  • {sp.upper():5s}: {n_img} ảnh, {n_da} mask lòng đường (drivable), {n_ll} mask vạch làn (lane)")

## 5. Tiến hành Huấn Luyện YOLOPv2 (Freeze Backbone 100%)
- **Chiến lược**: Đóng băng 100% E-ELAN Backbone (105 layer đầu), bảo toàn nguyên vẹn năng lực nhận diện của mạng gốc.
- **Mục tiêu**: Tối ưu hóa 2 Heads (Lane Line & Drivable Area) để nhận diện vạch kẻ đỏ sắc nét và làn xe xanh nước biển phẳng mịn.
- **Batch Size**: 16 (tối ưu tối đa tốc độ trên Tesla T4).
- **Learning Rate**: $3 \times 10^{-5}$ với AdamW.

In [ ]:
!python training/train_yolopv2.py \
    --weights weights/yolopv2.pt \
    --data data/dataset \
    --epochs-p1 15 \
    --epochs-p2 5 \
    --batch-size 16 \
    --lr 0.00003 \
    --device cuda:0 \
    --output-dir runs/train_tusimple

## 6. Hiển thị Biểu đồ Quá trình Hội tụ Loss & Metrics

In [ ]:
from IPython.display import Image, display
from pathlib import Path

loss_chart = Path('runs/train_tusimple/training_loss_curves.png')
if loss_chart.exists():
    print("--- BIỂU ĐỒ HỘI TỤ LOSS ĐA NHIỆM ---")
    display(Image(str(loss_chart)))

metrics_chart = Path('runs/train_tusimple/metrics_curves.png')
if metrics_chart.exists():
    print("--- BIỂU ĐỒ CHỈ SỐ PHÂN ĐOẠN IoU ---")
    display(Image(str(metrics_chart)))

## 7. Đánh giá Trực quan Kết quả Suy luận trên Ảnh Kiểm thử

In [ ]:
import sys
sys.path.insert(0, '/content')
import cv2
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from lane_detection.yolopv2_detector import YOLOPv2Detector

weights_best = Path('runs/train_tusimple/yolopv2_multidomain_best.pt')
if not weights_best.exists():
    weights_best = Path('weights/yolopv2_multidomain.pt')

detector = YOLOPv2Detector(
    duong_dan_weights=str(weights_best),
    device='cuda:0' if torch.cuda.is_available() else 'cpu'
)

# Chọn ảnh mẫu từ tập test để suy luận
test_images = list(Path('data/dataset/test/images').glob('*.*'))
if test_images:
    sample_path = str(test_images[0])
    for p in test_images:
        if 'vn_' in p.name:
            sample_path = str(p)
            break
    
    print(f"Thử nghiệm trực quan trên: {Path(sample_path).name}")
    img_bgr = cv2.imread(sample_path)
    ket_qua = detector.phat_hien(img_bgr)
    vis = detector.ve_ket_qua(img_bgr, ket_qua, ve_xe=False, ve_lan=True, ve_vung_lai=True)

    plt.figure(figsize=(12, 7))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f'Kết quả YOLOPv2 sau huấn luyện: {Path(sample_path).name}', fontsize=13, fontweight='bold')
    plt.axis('off')
    plt.show()

## 8. Tải file Trọng số & Kết quả Huấn luyện về Máy tính

In [ ]:
from google.colab import files
import shutil
from pathlib import Path

# Đồng bộ file weights tốt nhất sang weights/yolopv2_multidomain.pt
best_pt = Path('runs/train_tusimple/yolopv2_multidomain_best.pt')
target_pt = Path('weights/yolopv2_multidomain.pt')
if best_pt.exists():
    shutil.copy2(best_pt, target_pt)
    print(f"✓ Đã cập nhật {target_pt} từ checkpoint tốt nhất!")

print('🚀 Đang tải file trọng số về máy tính...')
files.download(str(target_pt))